# Document Question Answering System (RAG)

The goal of this notebook is simple. You give it a PDF, you ask a question, and it answers using what is actually written in that PDF.

A normal language model can only answer from what it picked up during training, so it knows nothing about your own files. RAG gets around that. It reads the document you give it, pulls out the passages that look most relevant to your question, and passes those passages to the model so the answer stays tied to your text instead of the model guessing from memory.

Here is the whole flow, start to finish:

1. **Document ingestion** reads each PDF and pulls out the plain text.
2. **Text chunking** cuts that text into short overlapping passages.
3. **Embedding creation** turns each passage into a vector that captures what it means.
4. **Vector database** stores those vectors so they can be searched fast.
5. **Query processing** turns your question into a vector the same way.
6. **Context retrieval** finds the passages sitting closest to the question.
7. **Answer generation** hands those passages to a language model, which writes the answer.

Everything runs locally and costs nothing. No API keys, no paid services. Just open it in Colab and run the cells from top to bottom.

## 1. Setup

First we install the libraries. Give it a minute or two the first time.

* `pypdf` pulls the text out of PDF files.
* `sentence-transformers` builds the embeddings.
* `faiss-cpu` is the vector store that does the similarity search.
* `transformers` runs the model that writes the answers.

In [1]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 32.3 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", DEVICE)

Running on: cuda


## 2. Document ingestion

Upload the PDFs you want to question. In Colab the cell below pops open a file picker. If you are running this somewhere other than Colab, just drop your PDFs into a folder called `docs` beside the notebook and it will pick them up from there.

In [3]:
DOC_DIR = "docs"
os.makedirs(DOC_DIR, exist_ok=True)

# In Colab this opens the upload dialog. Anywhere else it just reads the docs folder.
try:
    from google.colab import files
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open(os.path.join(DOC_DIR, name), "wb") as f:
            f.write(content)
except Exception:
    print("Not running in Colab, so reading whatever PDFs are already in the docs folder.")

pdf_paths = [os.path.join(DOC_DIR, f) for f in os.listdir(DOC_DIR) if f.lower().endswith(".pdf")]
print(f"Found {len(pdf_paths)} PDF file(s):")
for p in pdf_paths:
    print(" ", os.path.basename(p))

Saving Water Pollution_Sources and Causes.pdf to Water Pollution_Sources and Causes.pdf
Found 1 PDF file(s):
  Water Pollution_Sources and Causes.pdf


In [4]:
def load_pdf_text(path):
    reader = PdfReader(path)
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(pages)

documents = []
for path in pdf_paths:
    text = load_pdf_text(path)
    if text.strip():
        documents.append({"source": os.path.basename(path), "text": text})
    else:
        print("Heads up, no readable text found in", os.path.basename(path))

total_chars = sum(len(d["text"]) for d in documents)
print(f"Loaded {len(documents)} document(s), {total_chars:,} characters in total.")

Loaded 1 document(s), 6,909 characters in total.


## 3. Text chunking

A whole document is too big to hand to the model in one go, and scanning an entire page to find one sentence is wasteful. So we slice the text into short passages of roughly a hundred and twenty words each. Smaller passages help here, since they keep each fact on its own instead of burying it inside a long block of unrelated points.

The passages overlap a little on purpose. That overlap stops an idea from getting cut cleanly in two at a boundary, so a passage that begins partway through a thought still carries the words that came right before it.

In [5]:
def chunk_text(text, chunk_size=120, overlap=25):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        piece = " ".join(words[start:end])
        if piece.strip():
            chunks.append(piece)
        if end == len(words):
            break
        start += chunk_size - overlap
    return chunks

chunks = []          # passage text
chunk_sources = []   # file each passage came from
for doc in documents:
    for piece in chunk_text(doc["text"]):
        chunks.append(piece)
        chunk_sources.append(doc["source"])

print(f"Created {len(chunks)} chunks.")
if chunks:
    print("\nExample chunk:\n", chunks[0][:400], "...")

Created 10 chunks.

Example chunk:
 Water Pollution Introduction Water pollution refers to the contamination of water bodies such as rivers, lakes, oceans, groundwater, and reservoirs by harmful substances that adversely affect the quality of water, human health, and aquatic ecosystems. Pollutants may be physical, chemi cal, biological, or radioactive substances that alter the natural characteristics of water. According to the World ...


## 4. Embedding creation

Every chunk gets turned into a vector, which is just a list of numbers that stands in for its meaning. Passages about similar ideas end up with vectors pointing in similar directions, and that is what lets us search by meaning instead of matching exact words.

The model doing this is `all-MiniLM-L6-v2`. It is small, quick, and runs happily on a free Colab machine.

In [6]:
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)

chunk_embeddings = embedder.encode(
    chunks,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # normalizing lets us treat inner product as cosine similarity
).astype("float32")

print("Embedding matrix shape:", chunk_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape: (10, 384)


## 5. Vector database

FAISS keeps all the chunk vectors together and finds the closest ones to a query almost instantly. Since the vectors are already normalized, an inner product search lines up exactly with cosine similarity, so the top score is the passage that matches best.

In [7]:
dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)
print(f"Index built with {index.ntotal} vectors of dimension {dim}.")

Index built with 10 vectors of dimension 384.


## 6. Context retrieval

To answer a question we turn it into a vector the same way we did the chunks, then ask FAISS for the passages whose vectors are nearest. Those passages become the context the model reads before answering.

In [8]:
def retrieve(question, top_k=5):
    q_vec = embedder.encode(
        [question], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    scores, ids = index.search(q_vec, top_k)
    results = []
    for score, i in zip(scores[0], ids[0]):
        results.append({
            "text": chunks[i],
            "source": chunk_sources[i],
            "score": float(score),
        })
    return results

# quick sanity check on what comes back
for r in retrieve("What is this document about?"):
    print(f"[{r['source']}  score={r['score']:.3f}]")
    print(r["text"][:200], "...\n")

[Water Pollution_Sources and Causes.pdf  score=0.116]
Sources include: • Nuclear power plants • Medical facilities • Research laboratories Effects • Genetic mutations • Cancer • Long-term ecosystem damage 9. Thermal Pollution Caused by: • Power plants •  ...

[Water Pollution_Sources and Causes.pdf  score=0.081]
Water Pollution Introduction Water pollution refers to the contamination of water bodies such as rivers, lakes, oceans, groundwater, and reservoirs by harmful substances that adversely affect the qual ...

[Water Pollution_Sources and Causes.pdf  score=0.070]
animals releases nutrients. Effects: • Increased nutrient concentration • Reduction of dissolved oxygen • Growth of microorganisms 4. Natural Disasters Floods and landslides transport sediments and co ...

[Water Pollution_Sources and Causes.pdf  score=0.068]
marine ecosystems • Death of fish and birds • Reduced oxygen transfer • Toxic effects on marine life 6. Mining Activities Mining generates large quantities of waste m

## 7. Answer generation

The passages we retrieved get folded into a prompt and passed to `flan-t5-large`, a model tuned to follow instructions. It reads the context and writes the answer in its own words, staying grounded in the passages it was handed.

Two settings do most of the work for answer quality. We use beam search instead of plain greedy decoding, so the model weighs a few candidate answers and keeps the best one, and we set a minimum length so it writes a real sentence rather than a one word fragment like a stray list number.

This model is on the larger side, so switch on a GPU first under Runtime, then Change runtime type, and it will answer quickly. On a plain CPU it still runs, just slower. If you want something lighter, set `MODEL_NAME` back to `google/flan-t5-base`, though the answers get noticeably shorter.

In [9]:
MODEL_NAME = "google/flan-t5-large"   # large writes far better answers; set to flan-t5-base for a lighter run
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

def generate_answer(question, context):
    prompt = (
        "Answer the question in one or two complete sentences using only the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}"
    )
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=1024
    ).to(DEVICE)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=200,
        min_new_tokens=20,          # forces a full sentence instead of a one word reply
        num_beams=4,                # beam search picks a better answer than greedy decoding
        no_repeat_ngram_size=3,
        early_stopping=True,
    )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

print("Generator ready.")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generator ready.


## 8. The full pipeline

This is where everything comes together. Hand it a question and it retrieves the relevant passages, builds the context, writes a grounded answer, and shows you which parts of the document it leaned on.

In [10]:
def rag_answer(question, top_k=5, show_sources=True):
    hits = retrieve(question, top_k=top_k)
    context = "\n\n".join(h["text"] for h in hits)
    answer = generate_answer(question, context)

    print("Question:", question)
    print("\nAnswer:", answer)
    if show_sources:
        print("\nRetrieved from:")
        for h in hits:
            print(f"  - {h['source']} (score {h['score']:.3f})")
    return answer

## 9. Ask your document

The questions below are written for the water pollution PDF. Change them, or add your own, to fit whatever you uploaded.

In [11]:
rag_answer("What is water pollution?")

Question: What is water pollution?

Answer: The undesirable change in the physical, chemical, and biological characteristics of water that makes it harmful to living organisms and unsuitable for intended uses

Retrieved from:
  - Water Pollution_Sources and Causes.pdf (score 0.776)
  - Water Pollution_Sources and Causes.pdf (score 0.681)
  - Water Pollution_Sources and Causes.pdf (score 0.677)
  - Water Pollution_Sources and Causes.pdf (score 0.631)
  - Water Pollution_Sources and Causes.pdf (score 0.593)


'The undesirable change in the physical, chemical, and biological characteristics of water that makes it harmful to living organisms and unsuitable for intended uses'

In [12]:
rag_answer("What are the two main categories of sources of water pollution?")

Question: What are the two main categories of sources of water pollution?

Answer: Point Sources of Water Pollution Point sources are identifiable and localized sources from which pollutants are discharged directly into water bodies

Retrieved from:
  - Water Pollution_Sources and Causes.pdf (score 0.775)
  - Water Pollution_Sources and Causes.pdf (score 0.692)
  - Water Pollution_Sources and Causes.pdf (score 0.669)
  - Water Pollution_Sources and Causes.pdf (score 0.664)
  - Water Pollution_Sources and Causes.pdf (score 0.637)


'Point Sources of Water Pollution Point sources are identifiable and localized sources from which pollutants are discharged directly into water bodies'

In [13]:
rag_answer("What are some examples of point sources of water pollution?")

Question: What are some examples of point sources of water pollution?

Answer: Industrial discharge pipes • Municipal sewage outlets • Wastewater treatment plants • Oil refinery effluents • Power plants

Retrieved from:
  - Water Pollution_Sources and Causes.pdf (score 0.706)
  - Water Pollution_Sources and Causes.pdf (score 0.685)
  - Water Pollution_Sources and Causes.pdf (score 0.652)
  - Water Pollution_Sources and Causes.pdf (score 0.622)
  - Water Pollution_Sources and Causes.pdf (score 0.614)


'Industrial discharge pipes • Municipal sewage outlets • Wastewater treatment plants • Oil refinery effluents • Power plants'

In [14]:
rag_answer("Which heavy metals are found in industrial effluents?")

Question: Which heavy metals are found in industrial effluents?

Answer: (lead, mercury, cadmium, chromium) & Acids

Retrieved from:
  - Water Pollution_Sources and Causes.pdf (score 0.513)
  - Water Pollution_Sources and Causes.pdf (score 0.416)
  - Water Pollution_Sources and Causes.pdf (score 0.407)
  - Water Pollution_Sources and Causes.pdf (score 0.400)
  - Water Pollution_Sources and Causes.pdf (score 0.399)


'(lead, mercury, cadmium, chromium) & Acids'

In [15]:
rag_answer("Which gases are responsible for acid rain?")

Question: Which gases are responsible for acid rain?

Answer: Sulfur dioxide (SO2) • Nitrogen oxides (NOx) - NOx

Retrieved from:
  - Water Pollution_Sources and Causes.pdf (score 0.442)
  - Water Pollution_Sources and Causes.pdf (score 0.371)
  - Water Pollution_Sources and Causes.pdf (score 0.338)
  - Water Pollution_Sources and Causes.pdf (score 0.321)
  - Water Pollution_Sources and Causes.pdf (score 0.319)


'Sulfur dioxide (SO2) • Nitrogen oxides (NOx) - NOx'

In [16]:
# Type your own question here
rag_answer("How do agricultural activities cause water pollution?")

Question: How do agricultural activities cause water pollution?

Answer: a) Fertilizers Contain: • Nitrogen • Phosphorus • Potassium Effects: • Eutrophication • Algal blooms • Oxygen depletion

Retrieved from:
  - Water Pollution_Sources and Causes.pdf (score 0.722)
  - Water Pollution_Sources and Causes.pdf (score 0.686)
  - Water Pollution_Sources and Causes.pdf (score 0.683)
  - Water Pollution_Sources and Causes.pdf (score 0.652)
  - Water Pollution_Sources and Causes.pdf (score 0.629)


'a) Fertilizers Contain: • Nitrogen • Phosphorus • Potassium Effects: • Eutrophication • Algal blooms • Oxygen depletion'

## Improvements and experiments

What is here is kept deliberately simple. A few ways to take it further:

* **Chunking strategy.** Split on sentences or paragraphs instead of fixed word counts so passages fall on natural boundaries.
* **Stronger embeddings.** Bigger models like `bge-base-en-v1.5` or `gte-base` retrieve more accurately than MiniLM, though they run a little slower.
* **Hybrid search.** Pair this vector search with a keyword method such as BM25 so exact terms and rare names do not slip through.
* **Reranking.** Pull a wider set of passages first, then use a cross encoder to reorder them before answering.
* **A stronger generator.** Step up to `flan-t5-large`, or bring in a hosted model, when answer quality matters more than staying fully local.
* **Show confidence.** Surface the retrieval scores so a low top score can hint that the document probably does not hold the answer.

## Conclusion

This notebook walks through a full RAG pipeline from one end to the other: load the documents, break them into passages, embed and index those passages, retrieve the ones that fit a question, and generate an answer grounded in that retrieved text.

The same idea sits behind a lot of real tools, from documentation assistants to customer support bots to search across a company's own files. What makes RAG useful is that it lets a language model speak accurately about private or specialized material it never saw in training, just by feeding it the right context at the right moment.